# simple statistics in R for archaeologists

Now, Kevin Gratski has an excellent course on 'Digital Archaeology and Data Reuse', https://github.com/kgarstki/Introduction-Digital-Archaeology-Course and we are going to follow what he does with his class here:  https://github.com/kgarstki/ANTH-641_Stats-with-R

This document is a remix of that one. Make sure to read all comments carefully.

In [ ]:
# The "archdata" package was created by David L. Carlson and Georg Roth for
# "Quantitative Methods in Archaeology Using R" by David L. Carlson (2017).

install.packages("archdata")
install.packages(c("RcmdrMisc"), type="binary")

# you'll get some warnings, but you can ignore them. This might take a few moments.

Tip: you can go to [rdocumentation.org](https://www.rdocumentation.org/packages/archdata/versions/1.2-1) to see the information about all of the different archaeological datasets that have been packaged into 'archdata'. For instance, we're about to explore the 'DartPoints' dataset. Metadata about that dataset is directly behind [this link](https://www.rdocumentation.org/packages/archdata/versions/1.2-1/topics/DartPoints). Go have a look.

In [ ]:
# ok, tell R we want to use those two packages:
library(archdata)
library(RcmdrMisc)

# 1. Initial exploration for numerical and categorical data

Once you've collected data, you will want to explore the distribution of that data. We can look for the average values, or the most frequently occuring values (mean and median). We can look for how dispersed the data are around those central tendencies by looking for standard deviation or variance. Remember, we're trying to create some fragment of a past world from these observations; data don't tell a story, but they become something from which a story can be told.


In [ ]:
# Because the data is already packaged up for us in archdata, we can load it up with the 'data' command.
# Most of the time though we'd have to write a small function to read the data in from a csv file.
# Here we'll load up the 'DartPoints' dataframe:
data(DartPoints)
View(DartPoints) # notice that View starts with a capital V. Other commands don't. Annoying. Lower-case 'view' will give you an error.



In [ ]:
# Three ways of finding the average length of the dart points:
mean(DartPoints$Length)
mean(DartPoints[, 5])
mean(DartPoints[,"Length"])
# All three commands are equivalent. First, we tell R we want the mean, and then inside the brackets we
# tell it WHAT we want mean to be calculated on. In the first example, we give the name of the dataframe
# and then use the $ operator to specify which column by name. In the second, we give the name of the dataframe
# and then indicate the position that we want: the 5th column. The third specifies position, but uses the name of the column.


In [ ]:
# You can calculate each descriptive statistic one at a time, or you can get R to give you:
summary(DartPoints$Length)


In [ ]:
# So let's indicate the number of significant digits we want (places after the decimal) and then
# get R to give us a summary for *all* the columns, from the 5th position to the 11th:
options(digits=3)
numSummary(DartPoints[, 5:11])

In [ ]:
# What happens if you try to include columns 1 to 4?




In [ ]:
# How many points are there for each type of point? The type is in the 'Name' column.
# We make a new variable called DP_Type, and it gets the results of the command
# table(DartPoints$Name).

# The outer parentheses are a short cut to print the result of what goes inside DP_Type.
(DP_Type <- table(DartPoints$Name))

# We can work out the proportion each type has using prop.table on DP_Type, then multiplying by 100.
prop.table(DP_Type)*100

In [ ]:
# We found the average length earlier,
# but what does the distribution of lengths look like? Let's make a histogram:
hist(DartPoints$Length, main="Distribution of Dart Point Lengths", xlab="Length (mm)", col="grey")

# We know there are different types (Names) of dart points.
# Do they have different lengths? A boxplot is the best way to compare continuous
# data across different categories.
# The tilde (~) means "predicted by" or "broken down by".
# So Length ~ Name means "Show me Length broken down by Name"

boxplot(Length ~ Name, data=DartPoints,
        main="Dart Point Lengths by Typology",
        xlab="Point Type", ylab="Length (mm)",
        col="lightblue")

# Look at the graph. Do some point types seem generally longer than others?

In [ ]:
# Looking at the boxplot, the different point types seem to have different average lengths.
# But are these differences statistically significant, or just random noise in our sample?
# Because Length is continuous (numbers) and Name is categorical (text), we use an ANOVA test.

dp_aov <- aov(Length ~ Name, data=DartPoints)
summary(dp_aov)

# Look at the 'Pr(>F)' column. This is your p-value!
# Because it is TINY (way less than 0.05, shown by the ***), we can reject the null hypothesis.
# The point types definitively have different mean lengths.

# If you want to know exactly WHICH points are different from one another,
# you can run Tukey's test on our ANOVA results:
TukeyHSD(dp_aov)

In [ ]:
# What if we want to compare two continuous numeric measurements?
# For example, as a dart point gets longer, does it predictably get wider?
# We can visualize this with a scatterplot.

plot(DartPoints$Length, DartPoints$Width,
     main="Dart Point Length vs. Width",
     xlab="Length (mm)", ylab="Width (mm)",
     pch=19, col="darkblue") # pch=19 makes the dots solid circles

# Let's add a trend line to the plot to see the relationship:
abline(lm(Width ~ Length, data=DartPoints), col="red", lwd=2)

# It looks like there is a positive relationship. Let's test it statistically
# using a Correlation Test:

cor.test(DartPoints$Length, DartPoints$Width)

# The p-value tells us if the correlation is significant.
# The 'cor' value at the bottom (between -1 and 1) tells us how strong the relationship is.
# A value around 0.5 or 0.6 is a moderate positive correlation.

## Initial exploration for categorical data

In DartPoints, there's a lot of data about the kind of blade, and the shape of the blade. These are categories; something either is in one category, or another. So one thing we can do is see how they relate to each other by cross tabulation.

In [ ]:
# Here, we create a new variable to contain the results of the xtabs command. You can almost read
# the code like this: DP_CT gets the results of xtabs of Name versus Blade.Sh from DartPoints.

(DP_CT <- xtabs(~Name+Blade.Sh, DartPoints))

In [ ]:
# We can add up each row and column by using the addmargins command:
addmargins(DP_CT)

In [ ]:
# The bottom right hand corner shows us how many records we've added: 89. But do we have all the data? Let's check
# with the `dim` command (dimension):
dim(DartPoints)

In [ ]:
# that command tells us we've got 91 rows and 17 columns. So DP_CT is missing some data!
# What gives? If you look at DartPoints:
View(DartPoints)

In [ ]:
# you'll see we're missing cells of data. That might be the issue. So let's add the missing data
# to our cross tabulation:

addmargins(xtabs(~Name+addNA(Blade.Sh), DartPoints))

## YOUR TURN #1

Add another dataframe from archdata, and explore the numerical and categorical data. Write down any patterns you see. Remember, you can go to [https://www.rdocumentation.org/packages/archdata/versions/1.2-1](https://www.rdocumentation.org/packages/archdata/versions/1.2-1) to see what other data is there, and then load it here by name with the `data(name-goes-here)` command


In [ ]:
# put your stuff in here

# 2. Graphs
Looking at data with Graphs - Often, graphs provide us with a good summary of data and can help illuminate patterns.

In [ ]:

# We're going to look at two ways to visualize the number of coils present on La Tène fibulae
# from the Iron Age cemetery of Münsingen near Berne, Switzerland. Bring the dataframe into your directory.
data(Fibulae)
View(Fibulae)

In [ ]:
# We're interested in the variable "Coils"
(Fib_coils <- table(Fibulae$Coils))


In [ ]:
# First we'll make a pie chart of the number of coils. We'll include a main title and title our variable.
pie(Fib_coils, main="La Tene Bronze Fibulae", xlab="Number of Coils", clockwise=TRUE)
# Beautiful!

In [ ]:
# Let's compare it to a barplot of the same data.
barplot(Fib_coils, main="La Tene Bronze Fibulae", xlab="Number of Coils", ylab="Number of Fibulae")
# Which way to visualize the data is more helpful?

In [ ]:
# We'll try one more graph, using more complex data. First bring the dataframe Pithouses into the directory.
# This includes the desription of 45 Arctic Norway pithouses with 6 categorical variables.
data(PitHouses)

In [ ]:
# Then we'll create a cross-tabulation of Hearths with Size
(PitHouses.tbl <- xtabs(~Hearths+Size, PitHouses))

In [ ]:

# Then create a barplot of the crosstab. We'll include a legend that describes are different Hearths.
# Remember to export the grab when you're done.
barplot(PitHouses.tbl, ylab="Frequency", main="Arctic Norway Pithouses", beside=TRUE,legend.text=TRUE, args.legend=list(title="Hearths"))


## YOUR TURN #2.

Find a new dataset from the archdata package. Use one of these two create a graph of the data. You can use a scatterplot (plot), a bargraph (barplot), pie chart (pie), or boxplot (boxplot). You can save this file, and then use the View -> Command Palette -> PKM: Show Notebook Cell Overview to find the id of the cell with your scatterplot so that you can embed it into a markdown file, like eg your weekly log for this week.


In [ ]:
# put your stuff in here

# 3. Chi-squared test of association
see also https://www.r-bloggers.com/chi-squared-test/

Do two categories of data seem to 'go together'? And if they do, what might that mean? First we need a contingency table. This will allow us to say, is there an association between x and y?

In [ ]:

# Let's say we suspect there's something up with the form of the house and the depth of hearth.
# So we make a new table with just that data:

(table(PitHouses$Orient, PitHouses$Form))

In [ ]:
# Looking at the numbers, it kinda looks like _something_ is going on?
# But that's why we do the test.
#
# Chi-square measures the difference between what we observe in the
# contingency table, and what we would expect, if the distribution
# is normal.

# X² = Σ (O-E)² / E

# We form a 'null hypothesis': that there is no association. The question is, do the data allow us to reject the null?

# Pearson's Chi-squared test is simple to perform.


chisq.test(PitHouses$Orient, PitHouses$Form, correct=FALSE)

# The 'correct=FALSE' means, do we want to apply Yates' correction?
# (see https://en.wikipedia.org/wiki/Chi-squared_test for an accessible explanation)
#
# For now, we don't.

In [ ]:
# How do you know if the test has found an association?
# To make a decision using the p-value, you compare it to your alpha level (the significance threshold, which is usually 0.05)
# If it's greater than 0.05, we fail to reject the null hypothesis. (That is, we do not have enough evidence to conclude there is an association).
# If it's less than 0.05, we reject the null hypothesis of independence and conclude that the variables are likely associated


# Or, you can work out what the actual critical value of X2 is and compare the resulting X2 from your test against the critical value.
# Option 1: Asking for the 95th percentile
qchisq(p = 0.95, df = 1)

# Option 2: Asking for the top 5% directly
qchisq(p = 0.05, df = 1, lower.tail = FALSE)
# If X2 of your test is less than the critical value you fail to reject the null hypothesis
# If X2 of your test is greater than the critical value, we reject the null and conclude that there is a likely association



So: With regard to orientation versus form, is there any evidence of an association?

Remember, here's how to interpret the scores:

**Method 1: The Modern Way (using the p-value)**
* **P-value** = 0.9
* **Alpha** = 0.05
* **Conclusion:** Because 0.9 is greater than 0.05, we **fail to reject the null hypothesis**.

**Method 2: The Traditional Way (using the critical value)**
* **$X^2$ Test Statistic** = 0.007
* **Critical Value** = 3.841 *(for df = 1 at a 0.05 significance level)*
* **Conclusion:** Because 0.007 is less than 3.841, we **fail to reject the null hypothesis**.

### What does this result actually mean in plain English?
This result shows a massive lack of association between PitHouse Orientation and PitHouse Form.

* The **$X^2$ value of 0.007** is extremely close to zero. This means that the data you actually observed matches almost perfectly with what you would expect to see if the two variables had absolutely no connection to one another.
* The **p-value of 0.9** means that if the null hypothesis is true (no association), there is a 90% probability of seeing data distributed like this just by random chance.

Therefore, you have zero statistical evidence to suggest that the orientation of the pit houses is related to their form.

In [ ]:

# Now, when we look at the table of values, we see that oval houses with the gabel
# towards the coast only have 1 example. Chi-square is wonky if any of the values
# in a 2x2 table are less than 5, so we can't really conclude anything here.

# Let's see if there's an association between the number of hearths and the form of the house:

table(PitHouses$Hearths, PitHouses$Form)


In [ ]:
# Pearson's Chi-squared test
# What's the p value?

chisq.test(PitHouses$Hearths, PitHouses$Form, correct=FALSE)

In [ ]:
# Given the p-value what's the likelihood?
# Try to find the critical value for X2 - just re-use the code from earlier, but adjust for the degrees of freedom.

In [ ]:
# Let's try it on a different dataset, the EWBurials.
# Read up on the data by going to the documentation site, first
# Load and view the data:
data("EWBurials")
View(EWBurials)

In [ ]:
# We might imagine that there's an association between burial goods and the different groups:
table(EWBurials$Goods, EWBurials$Group)
# Pearson's Chi-squared test
chisq.test(EWBurials$Goods, EWBurials$Group, correct=FALSE)

# What is the p value? Is there evidence of an association
# What other associations might you imagine that we could test for?

In [ ]:
# Remember our problem about wonky results when counts are less than 5, from earlier? The Chi-square test gave us a warning because some
# of our expected cell values for PitHouses$Orient and PitHouses$Form were less than 5.
# When dealing with small archaeological samples, we have a specialized tool: Fisher's Exact Test.

fisher.test(PitHouses$Orient, PitHouses$Form)

# Look at the p-value! If it is greater than 0.05, we still accept the null hypothesis
# (no association), but this time we can do it with mathematical confidence despite our small sample.

## YOUR TURN #3

Explore some of the other datasets. See if you can find some associations.


In [ ]:
# put your stuff here

## YOUR TURN #4

Load up some graveyard data from past students. See what you can accomplish.


In [ ]:
library(curl)

graveyards <- read.csv( curl("https://raw.githubusercontent.com/shawngraham/hist3000/master/static/data/graveyards-data.csv"), sep=";" )

View(graveyards)

# Now... explore. Remember to add interesting things to your log for the week!